### Cell 1 — Install
This installs the `datasets` library, which we use to download the LEDGAR legal clause dataset from Hugging Face.

In [1]:
!pip install -q datasets

### Cell 2 — Settings
Here we set up the basic configuration for the project: which 10 types of legal clauses we want to classify, how many examples to use per category (150), and the minimum/maximum length a clause can have to be included.

In [2]:
import json
import random
from collections import Counter, defaultdict
from pathlib import Path
from datasets import load_dataset

RANDOM_SEED = 42
random.seed(RANDOM_SEED)

OUT_DIR = Path("data")
OUT_DIR.mkdir(parents=True, exist_ok=True)

CHOSEN_CATEGORIES = [
    "Governing Laws",
    "Terminations",
    "Confidentiality",
    "Indemnifications",
    "Assignments",
    "Notices",
    "Severability",
    "Counterparts",
    "Amendments",
    "Survival",
]

EXAMPLES_PER_CLASS = 150
MIN_CLAUSE_CHARS = 40
MAX_CLAUSE_CHARS = 2000

print(f"Config ready. Keeping {len(CHOSEN_CATEGORIES)} categories, "
      f"{EXAMPLES_PER_CLASS} examples each -> target {len(CHOSEN_CATEGORIES)*EXAMPLES_PER_CLASS} total.")

Config ready. Keeping 10 categories, 150 examples each -> target 1500 total.


### Cell 3 — Load the dataset
This downloads LEDGAR, a public dataset of ~80,000 real contract clauses from SEC filings, labeled into 100 categories. We only care about 10 of those categories, so we also set up a way to look up category names.

In [3]:
ledgar = load_dataset("coastalcph/lex_glue", "ledgar")

label_feature = ledgar["train"].features["label"]
id_to_name = {i: name for i, name in enumerate(label_feature.names)}
name_to_id = {name: i for i, name in id_to_name.items()}

print(f"LEDGAR loaded. Total categories available: {len(id_to_name)}")

# Sanity check: make sure every category we picked actually exists in LEDGAR
for cat in CHOSEN_CATEGORIES:
    assert cat in name_to_id, f"'{cat}' not found! Check spelling against: {label_feature.names}"
chosen_ids = {name_to_id[c] for c in CHOSEN_CATEGORIES}
print("All chosen categories verified present in LEDGAR.")

README.md:   0%|          | 0.00/34.1k [00:00<?, ?B/s]

ledgar/train-00000-of-00001.parquet:   0%|          | 0.00/20.9M [00:00<?, ?B/s]

ledgar/test-00000-of-00001.parquet:   0%|          | 0.00/3.31M [00:00<?, ?B/s]

ledgar/validation-00000-of-00001.parquet:   0%|          | 0.00/3.44M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/60000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

LEDGAR loaded. Total categories available: 100
All chosen categories verified present in LEDGAR.


### Cell 4 — Filter and balance the data
This goes through all the clauses, keeps only the ones belonging to our 10 chosen categories, removes duplicates and clauses that are too short or too long, and then picks exactly 150 clean examples per category. This keeps the dataset balanced so the model learns all 10 categories equally instead of favoring the most common one.

In [4]:
buckets = defaultdict(list)
for split in ["train", "validation", "test"]:
    for row in ledgar[split]:
        if row["label"] not in chosen_ids:
            continue
        text = row["text"].strip()
        if not (MIN_CLAUSE_CHARS <= len(text) <= MAX_CLAUSE_CHARS):
            continue
        buckets[row["label"]].append(text)

balanced = []  # list of (clause_text, category_name)
for lid, texts in buckets.items():
    unique_texts = list(dict.fromkeys(texts))  # dedup, keeps first-seen order
    random.shuffle(unique_texts)
    take = unique_texts[:EXAMPLES_PER_CLASS]
    name = id_to_name[lid]
    balanced.extend((t, name) for t in take)
    print(f"  {name:<18} available={len(unique_texts):>5}  taken={len(take)}")

random.shuffle(balanced)
print(f"\nTotal balanced examples collected: {len(balanced)}")

  Counterparts       available= 3344  taken=150
  Notices            available= 3232  taken=150
  Governing Laws     available= 4077  taken=150
  Assignments        available= 1692  taken=150
  Severability       available= 2545  taken=150
  Survival           available= 1930  taken=150
  Confidentiality    available= 1016  taken=150
  Amendments         available= 1814  taken=150
  Terminations       available= 1397  taken=150
  Indemnifications   available= 1047  taken=150

Total balanced examples collected: 1500


### Cell 5 — Format the data for training
Each clause is converted into a conversation format: an instruction explaining the task, the clause itself, and the correct category label. This is the format the model will actually learn from during fine-tuning.

In [5]:
CATEGORY_LIST_STR = ", ".join(CHOSEN_CATEGORIES)
SYSTEM_PROMPT = (
    "You are a legal clause classifier. Read the contract clause and respond "
    f"with exactly one category from this list: {CATEGORY_LIST_STR}. "
    "Respond with only the category name and nothing else."
)

def to_chat_record(clause_text: str, category: str) -> dict:
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Clause:\n{clause_text}"},
            {"role": "assistant", "content": category},
        ]
    }

records = [to_chat_record(t, c) for t, c in balanced]
print(f"Formatted {len(records)} records. Example:")
print(json.dumps(records[0], indent=2))

Formatted 1500 records. Example:
{
  "messages": [
    {
      "role": "system",
      "content": "You are a legal clause classifier. Read the contract clause and respond with exactly one category from this list: Governing Laws, Terminations, Confidentiality, Indemnifications, Assignments, Notices, Severability, Counterparts, Amendments, Survival. Respond with only the category name and nothing else."
    },
    {
      "role": "user",
      "content": "Clause:\nThis Agreement shall not have been terminated as to such Purchaser in accordance with Section 6.18 herein."
    },
    {
      "role": "assistant",
      "content": "Terminations"
    }
  ]
}


### Cell 6 — Split into train/validation/test
The data is split into three parts: 80% for training the model, 10% for checking progress during training, and 10% kept completely separate for final testing. The test set is never used during training — it's saved to give an honest, unbiased score at the end.

In [6]:
by_class = defaultdict(list)
for rec in records:
    by_class[rec["messages"][2]["content"]].append(rec)

train, val, test = [], [], []
for cat, recs in by_class.items():
    random.shuffle(recs)
    n = len(recs)
    a, b = int(0.8 * n), int(0.9 * n)
    train += recs[:a]
    val += recs[a:b]
    test += recs[b:]

random.shuffle(train); random.shuffle(val); random.shuffle(test)

print(f"Split sizes -> train={len(train)}  val={len(val)}  test={len(test)}")
print("Train label balance:", dict(Counter(r['messages'][2]['content'] for r in train)))

Split sizes -> train=1200  val=150  test=150
Train label balance: {'Survival': 120, 'Terminations': 120, 'Confidentiality': 120, 'Severability': 120, 'Assignments': 120, 'Amendments': 120, 'Indemnifications': 120, 'Governing Laws': 120, 'Counterparts': 120, 'Notices': 120}


### Cell 7 — Hard test cases
These are a few tricky clauses written by hand to test whether the model actually understands the clauses, rather than just matching keywords. For example, a clause about what survives after termination should be labeled "Survival," not "Terminations," even though it mentions the word "termination."

In [7]:
EDGE_CASES = [
    ("The provisions of Section 8 shall survive the expiration or termination "
     "of this Agreement and remain in full force.", "Survival"),
    ("Any notice of termination must be delivered in writing to the addresses "
     "below.", "Notices"),
    ("This Agreement, and any dispute arising hereunder, shall be governed by "
     "the internal laws of New York.", "Governing Laws"),
    ("The Company shall defend, indemnify, and hold the Executive harmless "
     "against any third-party claims.", "Indemnifications"),
    ("The parties may execute this instrument in any number of counterparts, "
     "including electronic copies.", "Counterparts"),
    ("No amendment to this Agreement shall be effective unless in writing and "
     "signed by both parties.", "Amendments"),
    ("Should any clause be found unenforceable, the balance of the Agreement "
     "shall continue in effect.", "Severability"),
    ("Neither party shall assign its rights hereunder without the prior written "
     "consent of the other party.", "Assignments"),
    ("The Receiving Party shall not disclose the Disclosing Party's trade "
     "secrets to any third party.", "Confidentiality"),
    ("This Agreement may be terminated by either party for convenience upon "
     "sixty days' notice.", "Terminations"),
    ("All confidential information shall be returned upon termination of this "
     "Agreement.", "Confidentiality"),
    ("Notices of assignment shall be governed by the laws of Delaware.",
     "Notices"),
]

edge_records = [to_chat_record(t, c) for t, c in EDGE_CASES]
print(f"Edge-case benchmark ready: {len(edge_records)} handcrafted examples "
      f"(you'll expand this to 30-50 by hand later).")

Edge-case benchmark ready: 12 handcrafted examples (you'll expand this to 30-50 by hand later).


### Cell 8 — Save the dataset
This saves the training, validation, test, and hard-test-case data as files, along with our category list, so they can be reused in the next step: training the model.

In [8]:
def write_jsonl(path: Path, rows: list):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"  wrote {len(rows):>5} -> {path}")

print("Saving files:")
write_jsonl(OUT_DIR / "train.jsonl", train)
write_jsonl(OUT_DIR / "val.jsonl", val)
write_jsonl(OUT_DIR / "test.jsonl", test)
write_jsonl(OUT_DIR / "benchmark_edgecases.jsonl", edge_records)

with open(OUT_DIR / "categories.json", "w") as f:
    json.dump({"categories": CHOSEN_CATEGORIES, "system_prompt": SYSTEM_PROMPT}, f, indent=2)
print(f"  wrote config -> {OUT_DIR / 'categories.json'}")

print("\nSTEP 01 COMPLETE. Dataset ready for Step 02 (training).")

Saving files:
  wrote  1200 -> data/train.jsonl
  wrote   150 -> data/val.jsonl
  wrote   150 -> data/test.jsonl
  wrote    12 -> data/benchmark_edgecases.jsonl
  wrote config -> data/categories.json

STEP 01 COMPLETE. Dataset ready for Step 02 (training).


In [9]:
!zip -r data.zip data

  adding: data/ (stored 0%)
  adding: data/benchmark_edgecases.jsonl (deflated 86%)
  adding: data/train.jsonl (deflated 83%)
  adding: data/categories.json (deflated 49%)
  adding: data/val.jsonl (deflated 82%)
  adding: data/test.jsonl (deflated 82%)
